# Dependency package

# Dataset Setting

# Claude

In [1]:
"""
BCI Competition IV-2a (官方 .gdf 格式)
Riemannian K-Means 特徵提取 + sklearn LDA 分類

相依套件：
    pip install mne numpy scipy scikit-learn
"""

import numpy as np
import mne
from scipy.signal import butter, filtfilt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


# ─────────────────────────────────────────────────────────────
# 1. GDF DATA LOADING  (官方 BCICIV-2a 格式)
# ─────────────────────────────────────────────────────────────

# BCICIV-2a 事件代碼對照表
EVENT_ID = {
    'left_hand':  '769',   # Class 1
    'right_hand': '770',   # Class 2
    'feet':       '771',   # Class 3
    'tongue':     '772',   # Class 4
}

target_keys = ['769', '770']

# 22 條 EEG channel 名稱（依照 BCICIV-2a 官方順序）
EEG_CHANNELS = [
    'EEG-Fz',
    'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5',
    'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4',
    'EEG-8', 'EEG-9', 'EEG-10', 'EEG-11',
    'EEG-Pz',
    'EEG-12', 'EEG-13', 'EEG-14', 'EEG-15', 'EEG-16'
]


def load_gdf(
    filepath: str,
    tmin: float = 0.5,
    tmax: float = 4.5,
    l_freq: float = 8.0,
    h_freq: float = 30.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    讀取官方 BCICIV-2a .gdf 檔案並切出 trial epochs。

    Parameters
    ----------
    filepath : str
        .gdf 檔路徑，例如 'A01T.gdf'（T=training, E=evaluation）
    tmin : float
        Cue 出現後幾秒開始擷取（預設 0.5s，避免視覺誘發電位）
    tmax : float
        Cue 出現後幾秒結束擷取（預設 4.5s）
    l_freq : float
        Bandpass 低截止頻率 (Hz)
    h_freq : float
        Bandpass 高截止頻率 (Hz)

    Returns
    -------
    X : np.ndarray, shape (n_trials, 22, n_times)
        Band-passed EEG epochs，僅保留 22 條 EEG（去除 EOG）
    y : np.ndarray, shape (n_trials,)
        Class labels 1–4
    """
    # ── 讀取原始 GDF ──────────────────────────────────────────
    raw = mne.io.read_raw_gdf(
        filepath,
        stim_channel='auto',
        preload=True,
        verbose=False
    )

    # ── 挑選 22 條 EEG（丟掉 3 條 EOG）────────────────────────
    # MNE 讀入後 channel type 可能都是 EEG；改用名稱過濾更穩
    raw.pick_channels(EEG_CHANNELS, ordered=True)

    # ── Bandpass filter ───────────────────────────────────────
    raw.filter(l_freq=l_freq, h_freq=h_freq, method='iir', verbose=False)

    # ── 取得 events ──────────────────────────────────────────
    events, _ = mne.events_from_annotations(raw, verbose=False)

    # 只保留 4 個 MI cue 事件；MNE 會把 GDF 事件代碼映射成字串
    # 需要先查出映射後的 event_id 數字
    _, ann_event_id = mne.events_from_annotations(raw, verbose=False)

    # 過濾出 769–772 對應的 key
    cue_event_id = {
        k: v for k, v in ann_event_id.items()
        if k in target_keys
    }

    if not cue_event_id:
        # 有些轉換版本直接存數字字串 '769' 等
        cue_event_id = {
            k: v for k, v in ann_event_id.items()
            if int(v) in EVENT_ID.values()
        }

    print(ann_event_id.items())

    if not cue_event_id:
        raise RuntimeError(
            f"找不到 MI cue 事件（769–772）。\n"
            f"檔案內事件代碼：{ann_event_id}\n"
            "請確認此檔案為 training/evaluation 組（非 rest 組）。"
        )

    # ── Epoching ─────────────────────────────────────────────
    epochs = mne.Epochs(
        raw,
        events,
        event_id=cue_event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,          # 不做 baseline correction（Riemannian 不需要）
        preload=True,
        verbose=False
    )

    X = epochs.get_data()                        # (n_trials, 22, n_times)

    # 把 MNE event_id 映射回 1–4 的 label
    event_code_to_label = {v: i + 1 for i, v in enumerate(sorted(EVENT_ID.values()))}
    y = np.array([
        event_code_to_label[cue_event_id[k]]
        for k in epochs.event_id
        for _ in range((epochs.events[:, 2] == epochs.event_id[k]).sum())
    ])
    # 更穩健的 label 提取
    y = np.array([event_code_to_label[e] for e in epochs.events[:, 2]])

    print(f"[load_gdf] {filepath}")
    print(f"           trials={X.shape[0]}, ch={X.shape[1]}, samples={X.shape[2]}")
    print(f"           fs={raw.info['sfreq']} Hz, window=[{tmin}, {tmax}]s")
    print(f"           class distribution: {np.bincount(y)[1:]}")
    return X, y


# ─────────────────────────────────────────────────────────────
# 2. SPD COVARIANCE MATRICES
# ─────────────────────────────────────────────────────────────

def compute_covariance(X: np.ndarray, regularize: float = 1e-6) -> np.ndarray:
    """
    計算每個 trial 的正規化協方差矩陣（SPD matrix）。

    使用公式：Σ = (X_c @ X_c.T) / (T - 1)
    再加 Tikhonov 正規化確保嚴格正定。

    X:      (n_trials, n_channels, n_times)
    return: (n_trials, n_channels, n_channels)
    """
    n_trials, n_ch, n_t = X.shape
    C = np.zeros((n_trials, n_ch, n_ch))

    for i in range(n_trials):
        Xi = X[i] - X[i].mean(axis=1, keepdims=True)   # zero-mean
        Ci = (Xi @ Xi.T) / (n_t - 1)
        C[i] = Ci + regularize * np.eye(n_ch)

    return C


# ─────────────────────────────────────────────────────────────
# 3. RIEMANNIAN GEOMETRY  (純 NumPy)
# ─────────────────────────────────────────────────────────────

def _eigh_safe(A: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """eigen-decomposition，eigenvalue clip 確保正定"""
    vals, vecs = np.linalg.eigh(A)
    vals = np.maximum(vals, 1e-12)
    return vals, vecs


def matrix_sqrt(A: np.ndarray) -> np.ndarray:
    """A^{1/2}"""
    v, Q = _eigh_safe(A)
    return Q @ np.diag(np.sqrt(v)) @ Q.T


def matrix_sqrt_inv(A: np.ndarray) -> np.ndarray:
    """A^{-1/2}"""
    v, Q = _eigh_safe(A)
    return Q @ np.diag(1.0 / np.sqrt(v)) @ Q.T


def matrix_log(A: np.ndarray) -> np.ndarray:
    """對稱正定矩陣的對數 log(A)"""
    v, Q = _eigh_safe(A)
    return Q @ np.diag(np.log(v)) @ Q.T


def matrix_exp(A: np.ndarray) -> np.ndarray:
    """對稱矩陣的指數 exp(A)"""
    v, Q = np.linalg.eigh(A)          # exp 不需要 clip
    return Q @ np.diag(np.exp(v)) @ Q.T


def riemannian_distance(A: np.ndarray, B: np.ndarray) -> float:
    """
    Affine-invariant Riemannian distance：
        d(A, B) = || log( A^{-1/2} B A^{-1/2} ) ||_F
    """
    A_invsqrt = matrix_sqrt_inv(A)
    M = A_invsqrt @ B @ A_invsqrt
    logM = matrix_log(M)
    return float(np.sqrt(np.sum(logM ** 2)))


def riemannian_mean(
    C: np.ndarray,
    max_iter: int = 50,
    tol: float = 1e-7
) -> np.ndarray:
    """
    Fréchet (Karcher) mean on the SPD manifold。

    迭代公式：
        S   = (1/N) Σ log( M^{-1/2} C_i M^{-1/2} )
        M   ← M^{1/2} exp(S) M^{1/2}

    收斂條件：||M_new - M||_F / ||M||_F < tol
    """
    n = len(C)
    M = C.mean(axis=0)                  # Euclidean mean 作初始值

    for _ in range(max_iter):
        M_sqrt    = matrix_sqrt(M)
        M_invsqrt = matrix_sqrt_inv(M)

        S = np.zeros_like(M)
        for i in range(n):
            inner = M_invsqrt @ C[i] @ M_invsqrt
            S += matrix_log(inner)
        S /= n

        M_new = M_sqrt @ matrix_exp(S) @ M_sqrt
        delta = np.linalg.norm(M_new - M, 'fro') / (np.linalg.norm(M, 'fro') + 1e-12)
        M = M_new
        if delta < tol:
            break

    return M


# ─────────────────────────────────────────────────────────────
# 4. RIEMANNIAN K-MEANS
# ─────────────────────────────────────────────────────────────

def riemannian_kmeans(
    C: np.ndarray,
    n_clusters: int = 8,
    max_iter: int = 30,
    n_init: int = 3,
    seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """
    K-Means on SPD manifold。

    初始化：K-Means++ on Riemannian distance
    M-step：每個 cluster 取 Riemannian mean (Karcher flow)
    E-step：每個點分配到最近的 centroid

    Returns
    -------
    centroids : (K, p, p)
    labels    : (n_trials,)
    """
    rng = np.random.default_rng(seed)
    n   = len(C)
    best_inertia   = np.inf
    best_centroids = None
    best_labels    = None

    for run in range(n_init):
        # ── K-Means++ init ────────────────────────────────────
        idx = [int(rng.integers(n))]
        for _ in range(1, n_clusters):
            dists_sq = np.array([
                min(riemannian_distance(C[i], C[j]) for j in idx) ** 2
                for i in range(n)
            ])
            probs = dists_sq / dists_sq.sum()
            idx.append(int(rng.choice(n, p=probs)))
        centroids = C[np.array(idx)].copy()

        labels = np.zeros(n, dtype=int)
        for it in range(max_iter):

            # ── E-step ────────────────────────────────────────
            dist_mat   = np.array([
                [riemannian_distance(C[i], centroids[k]) for k in range(n_clusters)]
                for i in range(n)
            ])
            new_labels = dist_mat.argmin(axis=1)

            converged = np.all(new_labels == labels) and it > 0
            labels    = new_labels

            # ── M-step ────────────────────────────────────────
            new_centroids = centroids.copy()
            for k in range(n_clusters):
                members = C[labels == k]
                if len(members) == 1:
                    new_centroids[k] = members[0]
                elif len(members) > 1:
                    new_centroids[k] = riemannian_mean(members)
                # len==0: centroid 保持不變（空 cluster）
            centroids = new_centroids

            if converged:
                break

        inertia = float(sum(
            riemannian_distance(C[i], centroids[labels[i]]) ** 2
            for i in range(n)
        ))
        print(f"  [K-Means run {run+1}/{n_init}] inertia = {inertia:.6f}")

        if inertia < best_inertia:
            best_inertia   = inertia
            best_centroids = centroids.copy()
            best_labels    = labels.copy()

    return best_centroids, best_labels


# ─────────────────────────────────────────────────────────────
# 5. FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────

def extract_features(
    C: np.ndarray,
    centroids: np.ndarray
) -> np.ndarray:
    """
    每個 trial 到各 centroid 的 Riemannian 距離組成特徵向量。

    C:         (n_trials, p, p)
    centroids: (K, p, p)
    return:    (n_trials, K)
    """
    n, K = len(C), len(centroids)
    features = np.zeros((n, K))
    for i in range(n):
        for k in range(K):
            features[i, k] = riemannian_distance(C[i], centroids[k])
    return features


# ─────────────────────────────────────────────────────────────
# 6. MAIN PIPELINE
# ─────────────────────────────────────────────────────────────

def run_pipeline(
    train_gdf: str,
    eval_gdf:  str  = None,
    n_clusters: int = 8,
    n_cv_folds: int = 5,
    seed: int = 42
):
    """
    完整 pipeline：GDF 讀取 → 協方差 → K-Means → 特徵 → LDA

    Parameters
    ----------
    train_gdf : str
        Training 檔案，例如 'A01T.gdf'
    eval_gdf : str, optional
        Evaluation 檔案，例如 'A01E.gdf'
        若提供則額外做 train→eval 的獨立測試
    n_clusters : int
        Riemannian K-Means 的 K 值
    n_cv_folds : int
        Training set 上的 cross-validation folds
    """
    sep = "─" * 55
    print(f"\n{'═'*55}")
    print("  BCI IV-2a  |  Riemannian K-Means + LDA")
    print(f"{'═'*55}\n")

    # ── Load training data ────────────────────────────────────
    print("[1/5] Loading training GDF...")
    X_train, y_train = load_gdf(train_gdf)

    # ── Covariance ───────────────────────────────────────────
    print("\n[2/5] Computing SPD covariance matrices...")
    C_train = compute_covariance(X_train, regularize=1e-6)
    min_eig = np.linalg.eigvalsh(C_train).min()
    print(f"      SPD check — min eigenvalue: {min_eig:.3e}  ({'OK' if min_eig > 0 else 'FAIL'})")

    # ── Riemannian K-Means ───────────────────────────────────
    print(f"\n[3/5] Riemannian K-Means (K={n_clusters}, n_init=3)...")
    centroids, _ = riemannian_kmeans(C_train, n_clusters=n_clusters, seed=seed)

    # ── Feature extraction ───────────────────────────────────
    print("\n[4/5] Extracting Riemannian distance features...")
    F_train = extract_features(C_train, centroids)
    print(f"      Feature matrix: {F_train.shape}")

    # ── LDA cross-validation on training set ─────────────────
    print(f"\n[5/5] LDA — {n_cv_folds}-fold stratified CV on training set...")
    le = LabelEncoder()
    y_enc = le.fit_transform(y_train)

    lda = LinearDiscriminantAnalysis(solver='svd')
    cv  = StratifiedKFold(n_splits=n_cv_folds, shuffle=True, random_state=seed)
    scores = cross_val_score(lda, F_train, y_enc, cv=cv, scoring='accuracy')

    print(f"\n{sep}")
    print(f"  Training CV accuracy : {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")
    print(f"  Per-fold             : {[f'{s*100:.1f}%' for s in scores]}")
    print(f"  Chance level (4-cls) : 25.00%")
    print(sep)

    # Fit LDA on all training data
    lda.fit(F_train, y_enc)

    # ── Optional: evaluate on eval set ───────────────────────
    if eval_gdf is not None:
        print(f"\n[+] Evaluating on {eval_gdf}...")
        X_eval, y_eval = load_gdf(eval_gdf)
        C_eval = compute_covariance(X_eval, regularize=1e-6)
        F_eval = extract_features(C_eval, centroids)     # 用 training centroids
        y_eval_enc = le.transform(y_eval)
        acc = lda.score(F_eval, y_eval_enc)
        print(f"\n{sep}")
        print(f"  Evaluation set accuracy : {acc*100:.2f}%")
        print(sep)

    return lda, centroids, F_train, y_enc, scores


# ─────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # 單 subject training only
    lda, centroids, features, y, scores = run_pipeline(
        train_gdf  = "..\..\dataset\BCICIV_2a_gdf\A01T.gdf",
        # eval_gdf   = "A01E.gdf",   # 可選，沒有的話移除這行
        n_clusters = 8,
        n_cv_folds = 5,
        seed       = 42
    )


═══════════════════════════════════════════════════════
  BCI IV-2a  |  Riemannian K-Means + LDA
═══════════════════════════════════════════════════════

[1/5] Loading training GDF...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
dict_items([(np.str_('1023'), 1), (np.str_('1072'), 2), (np.str_('276'), 3), (np.str_('277'), 4), (np.str_('32766'), 5), (np.str_('768'), 6), (np.str_('769'), 7), (np.str_('770'), 8), (np.str_('771'), 9), (np.str_('772'), 10)])


KeyError: 7